In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install torch transformers datasets trl accelerate sentencepiece protobuf
!pip install --upgrade huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-la33xl1v/unsloth_b68936d60e754088823c6ac05ffa296e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-la33xl1v/unsloth_b68936d60e754088823c6ac05ffa296e
  Resolved https://github.com/unslothai/unsloth.git to commit c356427f30e8abadd38fbc722e494321f0b5b64b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 16.2 MB/s eta 0:00:00
   

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Adjust based on your dataset length
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to save memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B", # Or your specific FunctionGemma base model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/393M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/13.9k [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any rank like 8, 16, 32, 64
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "lm_head", "embed_tokens"], # Including these helps with structural text tokens
    lora_alpha = 32,
    lora_dropout = 0, # Optimized to 0
    bias = "none",    # Optimized to none
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [ ]:
from datasets import load_dataset

# Load your custom single-turn JSONL dataset
dataset = load_dataset("json", data_files="comprehensive_math_dataset.jsonl", split="train")

# Standard Unsloth chat template formatting utility
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma", # Forces Gemma's native token mapping
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
)

def formatting_prompts_func(examples):
    convs = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convs]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Verified False for distinct conversational sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        max_steps = 400, # Adjust based on dataset scale, usually 1-3 epochs is fine
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        save_strategy = "no",               # Prevents the trainer from crashing mid-run
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 400
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 3,796,992 of 271,895,168 (1.40% trained)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard

Step,Training Loss
1,3.439749
2,3.401618
3,3.319434
4,2.943533
5,2.547903
6,2.325124
7,2.015721
8,1.829621
9,1.736835
10,1.664356


In [ ]:
# Save LoRA adapters
model.save_pretrained_merged("gemma_function_router", tokenizer, save_method = "lora")

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in gemma_function_router/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:09<00:00, 10.00s/it]


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Downloaded tokenizer.model


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:03<00:00,  3.97s/it]


Unsloth: Merge process complete. Saved to `/content/gemma_function_router`


In [ ]:
import os
from google.colab import files

# 1. Zip the folder
!zip -r gemma_function_router.zip /content/gemma_function_router

# 2. Trigger the download to your local machine
files.download('gemma_function_router.zip')

  adding: content/gemma_function_router/ (stored 0%)
  adding: content/gemma_function_router/chat_template.jinja (deflated 61%)
  adding: content/gemma_function_router/model.safetensors (deflated 21%)
  adding: content/gemma_function_router/tokenizer.model (deflated 52%)
  adding: content/gemma_function_router/tokenizer_config.json (deflated 97%)
  adding: content/gemma_function_router/.cache/ (stored 0%)
  adding: content/gemma_function_router/.cache/huggingface/ (stored 0%)
  adding: content/gemma_function_router/.cache/huggingface/CACHEDIR.TAG (deflated 24%)
  adding: content/gemma_function_router/.cache/huggingface/.gitignore (stored 0%)
  adding: content/gemma_function_router/.cache/huggingface/download/ (stored 0%)
  adding: content/gemma_function_router/.cache/huggingface/download/tokenizer.model.metadata (deflated 29%)
  adding: content/gemma_function_router/.cache/huggingface/download/model.safetensors.metadata (deflated 31%)
  adding: content/gemma_function_router/tokenizer.j

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,                  # Scaled up slightly for a longer run (~3% of total steps)
        num_train_epochs = 3,               # <-- Changed from max_steps to 3 full epochs
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,                  # Changed to 5 so your logs don't spam 1,875 lines
        output_dir = "outputs",

        # --- THE PICKLING ERROR FIX ---
        save_strategy = "no",               # Prevents the trainer from crashing mid-run
        report_to = "none",                 # Keeps things clean unless you use Wandb
    ),
)

# Start training
trainer_stats = trainer.train()

